In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.svm import OneClassSVM
from scipy.stats import mode
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers
from sklearn.neighbors import KernelDensity
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from scipy.stats import spearmanr
import os
import pandas as pd
import numpy as np
from scipy.signal import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.spatial.distance import cdist
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import numpy as np
from scipy.signal import butter, sosfilt
from scipy.signal import resample
from scipy.fft import fft
import os
import numpy as np
import pandas as pd
from scipy.linalg import logm
from sklearn.neighbors import KernelDensity
from scipy.special import logsumexp
import numpy as np
import scipy
import scipy.signal
import time
import tracemalloc
import umap
from sklearn.decomposition import PCA
from sklearn.decomposition import KernelPCA
from sklearn.manifold import Isomap, LocallyLinearEmbedding
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
import pickle
import os

import numpy as np
import random

np.random.seed(1)
random.seed(1)

### Load

In [43]:
import pandas as pd

def load_dataset(path):
    """
    Minimal CSV loader.
    Returns:
        df : pandas.DataFrame
    """
    df = pd.read_csv(path)
    return df


In [44]:
df = load_dataset(path="../../Datasets/MotorImagery/processed/bci_features.csv",)
df.head()

,subject,session,label,time_0_0,time_0_1,time_0_2,time_0_3,time_0_4,time_0_5,time_0_6,...,freq_18_18,freq_18_19,freq_18_20,freq_18_21,freq_19_19,freq_19_20,freq_19_21,freq_20_20,freq_20_21,freq_21_21
0,A01,session1,3,5.267527e-11,4.034399e-11,4.678092e-11,4.988477e-11,4.692093e-11,4.089548e-11,1.548455e-11,...,1.046879e-10,5.565342e-11,7.896886e-12,5.339151e-12,1.311974e-10,4.392987e-11,8.063179e-11,8.457327e-11,1.187630e-10,2.480122e-10
1,A01,session1,2,4.258623e-11,3.805907e-11,4.301141e-11,4.352206e-11,4.202700e-11,3.613776e-11,2.298732e-11,...,9.521127e-11,5.115321e-11,2.080679e-11,2.464693e-11,1.135982e-10,3.985870e-11,7.787790e-11,7.793657e-11,1.220109e-10,2.743620e-10
2,A01,session1,1,3.245072e-11,2.707170e-11,3.160530e-11,3.281442e-11,3.181356e-11,2.884676e-11,1.674429e-11,...,5.976102e-11,3.050644e-11,6.442204e-12,1.171661e-11,5.464778e-11,1.457287e-11,2.367965e-11,3.652671e-11,5.083133e-11,1.182691e-10
3,A01,session1,0,5.061837e-11,4.290988e-11,4.819076e-11,4.964892e-11,4.569411e-11,3.962393e-11,2.052034e-11,...,8.840444e-11,2.861593e-11,1.735132e-11,2.208399e-11,9.064836e-11,3.220702e-11,4.230613e-11,8.441786e-11,1.219668e-10,2.350288e-10
4,A01,session1,0,2.911876e-11,2.147890e-11,2.556846e-11,2.738989e-11,2.745035e-11,2.583514e-11,1.027371e-11,...,7.598676e-11,3.177676e-11,2.540148e-11,2.644370e-11,7.674789e-11,3.823329e-11,5.740674e-11,7.025289e-11,1.028704e-10,2.139844e-10


### Feature Selection

Selection

In [ ]:
import numpy as np

from sklearn.feature_selection import (
    mutual_info_classif, f_classif, chi2,
    SequentialFeatureSelector, RFE
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import NearestNeighbors
from scipy.stats import spearmanr


# ==========================================================
# FS: none
# ==========================================================
def fs_none(X_train, y_train, domains_train, X_test, domains_test, **kwargs):
    return X_train, X_test, {"selected_features": None}


# ==========================================================
# 1) Mutual Information
# ==========================================================
def fs_mi(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    mi = mutual_info_classif(X_train, y_train, discrete_features=False)
    idx = np.argsort(mi)[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 2) ANOVA F-test
# ==========================================================
def fs_anova(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    f_vals, _ = f_classif(X_train, y_train)
    f_vals = np.nan_to_num(f_vals, nan=-np.inf, posinf=-np.inf, neginf=-np.inf)
    idx = np.argsort(f_vals)[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 3) Variance
# ==========================================================
def fs_variance(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    vars_ = np.var(X_train, axis=0)
    idx = np.argsort(vars_)[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 4) Pearson correlation (abs)
# ==========================================================
def fs_pearson(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    corrs = np.array([np.corrcoef(X_train[:, i], y_train)[0, 1] for i in range(X_train.shape[1])])
    corrs = np.nan_to_num(corrs)
    idx = np.argsort(np.abs(corrs))[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 5) Chi-squared (shift to nonnegative; shift based on TRAIN only)
# ==========================================================
def fs_chi2(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    shift = X_train.min(axis=0)
    X_train_nn = X_train - shift
    X_test_nn = X_test - shift

    chi2_vals, _ = chi2(X_train_nn, y_train)
    chi2_vals = np.nan_to_num(chi2_vals, nan=-np.inf, posinf=-np.inf, neginf=-np.inf)
    idx = np.argsort(chi2_vals)[::-1][:n_features]

    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 6) Relief (binary only) – uses TRAIN only for scoring
# ==========================================================
def fs_relief(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    classes = np.unique(y_train)
    if len(classes) != 2:
        raise ValueError("Relief here supports only binary classification.")

    # normalize using TRAIN only (no leakage)
    x_min = X_train.min(axis=0)
    x_ptp = X_train.ptp(axis=0) + 1e-12
    Xn = (X_train - x_min) / x_ptp

    nbrs = NearestNeighbors(n_neighbors=2).fit(Xn)
    _, idxs = nbrs.kneighbors(Xn)

    scores = np.zeros(X_train.shape[1], dtype=float)
    for i, (x_i, y_i) in enumerate(zip(Xn, y_train)):
        nn_idx = idxs[i][1]
        x_nn = Xn[nn_idx]
        if y_train[nn_idx] == y_i:
            scores -= np.abs(x_i - x_nn)
        else:
            scores += np.abs(x_i - x_nn)

    idx = np.argsort(scores)[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 7) L1 Logistic (sparse)
# ==========================================================
def fs_l1(X_train, y_train, domains_train, X_test, domains_test, n_features=128, C=0.1):
    model = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=2000)
    model.fit(X_train, y_train)
    scores = np.abs(model.coef_).ravel()
    idx = np.argsort(scores)[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 8) Random Forest importance
# ==========================================================
def fs_rf(X_train, y_train, domains_train, X_test, domains_test, n_features=128, n_estimators=200):
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    importances = np.nan_to_num(model.feature_importances_)
    idx = np.argsort(importances)[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 9) Spearman correlation (abs)
# ==========================================================
def fs_spearman(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    corrs = np.array([spearmanr(X_train[:, i], y_train).correlation for i in range(X_train.shape[1])])
    corrs = np.nan_to_num(corrs)
    idx = np.argsort(np.abs(corrs))[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 10) Entropy-based relevance (your IG-like heuristic)
# ==========================================================
def fs_entropy(X_train, y_train, domains_train, X_test, domains_test, n_features=128, bins=10):
    def entropy_1d(a):
        hist, _ = np.histogram(a, bins=bins, density=True)
        p = hist[hist > 0]
        return -np.sum(p * np.log(p))

    H_y = entropy_1d(y_train)
    ig = []
    for i in range(X_train.shape[1]):
        H_x = entropy_1d(X_train[:, i])
        H_xy = entropy_1d(np.concatenate([X_train[:, i], y_train]))
        ig.append(H_x + H_y - H_xy)

    ig = np.nan_to_num(np.array(ig))
    idx = np.argsort(ig)[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 11) mRMR (simple)
# ==========================================================
def fs_mrmr(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    mi = mutual_info_classif(X_train, y_train, discrete_features=False)
    selected = []
    remaining = list(range(X_train.shape[1]))

    while len(selected) < n_features and remaining:
        scores = []
        for f in remaining:
            if selected:
                redundancy = np.mean([np.corrcoef(X_train[:, f], X_train[:, s])[0, 1] for s in selected])
                redundancy = 0.0 if np.isnan(redundancy) else redundancy
            else:
                redundancy = 0.0
            scores.append(mi[f] - redundancy)

        best = remaining[int(np.argmax(scores))]
        selected.append(best)
        remaining.remove(best)

    idx = np.array(selected, dtype=int)
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 12) OneR (simple)
# ==========================================================
def fs_oner(X_train, y_train, domains_train, X_test, domains_test, n_features=128, bins=10):
    y_train = np.asarray(y_train).astype(int)
    scores = []

    for i in range(X_train.shape[1]):
        feature = X_train[:, i]
        if np.all(feature == feature[0]):
            scores.append(0.0)
            continue

        cuts = np.linspace(np.min(feature), np.max(feature), bins + 1)
        digitized = np.digitize(feature, cuts)

        preds = np.zeros_like(y_train)
        for b in np.unique(digitized):
            mask = digitized == b
            if np.any(mask):
                preds[mask] = np.argmax(np.bincount(y_train[mask]))
        scores.append(np.mean(preds == y_train))

    scores = np.nan_to_num(np.array(scores))
    idx = np.argsort(scores)[::-1][:n_features]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 13) SFS (Sequential Forward Selection with Logistic)
# ==========================================================
def fs_sfs(X_train, y_train, domains_train, X_test, domains_test, n_features=128, max_iter=1000):
    n_features = min(n_features, X_train.shape[1])
    model = LogisticRegression(max_iter=max_iter)

    sfs = SequentialFeatureSelector(
        model,
        n_features_to_select=n_features,
        direction="forward",
        n_jobs=-1
    )
    sfs.fit(X_train, y_train)
    idx = np.where(sfs.get_support())[0]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 14) SVM-RFE
# ==========================================================
def fs_svmrfe(X_train, y_train, domains_train, X_test, domains_test, n_features=128):
    n_features = min(n_features, X_train.shape[1])
    svm = SVC(kernel="linear")
    rfe = RFE(estimator=svm, n_features_to_select=n_features, step=1)
    rfe.fit(X_train, y_train)
    idx = np.where(rfe.get_support())[0]
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 15) GA feature selection (can be slow; defaults are modest)
# ==========================================================
def fs_ga(
    X_train, y_train, domains_train, X_test, domains_test,
    n_features=128, pop_size=20, generations=10, mutation_rate=0.05, cv=3, random_state=42
):
    rng = np.random.RandomState(random_state)
    n_total = X_train.shape[1]
    clf = LogisticRegression(max_iter=1000)

    pop = rng.randint(0, 2, size=(pop_size, n_total))

    def fitness(mask):
        k = int(mask.sum())
        if k == 0:
            return 0.0
        X_sel = X_train[:, mask == 1]
        # tiny subsets can be unstable; still ok
        return float(np.mean(cross_val_score(clf, X_sel, y_train, cv=cv, n_jobs=-1)))

    for _ in range(generations):
        scores = np.array([fitness(ind) for ind in pop])

        # keep top half
        idx = scores.argsort()[::-1][: max(2, pop_size // 2)]
        parents = pop[idx]

        # crossover
        children = []
        while len(children) < pop_size - len(parents):
            p1 = parents[rng.randint(len(parents))]
            p2 = parents[rng.randint(len(parents))]
            cross = rng.randint(1, n_total - 1)
            child = np.concatenate([p1[:cross], p2[cross:]])
            children.append(child)
        children = np.array(children)

        # mutation
        mut = rng.rand(*children.shape) < mutation_rate
        children = np.logical_xor(children, mut).astype(int)

        pop = np.vstack([parents, children])

    scores = np.array([fitness(ind) for ind in pop])
    best_mask = pop[int(scores.argmax())].astype(int)

    selected = np.where(best_mask == 1)[0]

    # enforce exactly n_features if too many selected (tie-break by variance)
    if len(selected) > n_features:
        vars_ = np.var(X_train[:, selected], axis=0)
        selected = selected[np.argsort(vars_)[::-1][:n_features]]

    # if too few selected, pad with top-variance among remaining
    if len(selected) < min(n_features, n_total):
        remaining = np.setdiff1d(np.arange(n_total), selected)
        vars_rem = np.var(X_train[:, remaining], axis=0)
        need = min(n_features - len(selected), len(remaining))
        add = remaining[np.argsort(vars_rem)[::-1][:need]]
        selected = np.concatenate([selected, add])

    idx = np.array(selected, dtype=int)
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}


# ==========================================================
# 16) CFS (your merit-based implementation)
# ==========================================================
def fs_cfs(X_train, y_train, domains_train, X_test, domains_test, k=32):
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)

    n_samples, n_features_total = X_train.shape
    k = min(k, n_features_total)

    y0 = y_train - y_train.mean()
    X0 = X_train - X_train.mean(axis=0)

    r_fc = np.abs((X0 * y0[:, None]).mean(axis=0) / (X0.std(axis=0) * y0.std() + 1e-8))
    r_fc = np.nan_to_num(r_fc)

    corr_ff = np.abs(np.corrcoef(X_train, rowvar=False))
    corr_ff = np.nan_to_num(corr_ff)

    selected = []
    remaining = np.arange(n_features_total)

    sum_r_cf = 0.0
    sum_r_ff = 0.0

    while len(selected) < k and len(remaining) > 0:
        best_feat = None
        best_score = -np.inf

        for f in remaining:
            kc = len(selected) + 1
            new_sum_r_cf = sum_r_cf + r_fc[f]

            if selected:
                new_sum_r_ff = sum_r_ff + 2.0 * np.sum(corr_ff[f, selected])
            else:
                new_sum_r_ff = 0.0

            r_cf_mean = new_sum_r_cf / kc
            r_ff_mean = new_sum_r_ff / (kc * (kc - 1) + 1e-8)

            merit = (kc * r_cf_mean) / np.sqrt(kc + kc * (kc - 1) * r_ff_mean + 1e-8)

            if merit > best_score:
                best_score = merit
                best_feat = int(f)
                best_sum_r_cf = new_sum_r_cf
                best_sum_r_ff = new_sum_r_ff

        selected.append(best_feat)
        remaining = remaining[remaining != best_feat]
        sum_r_cf = best_sum_r_cf
        sum_r_ff = best_sum_r_ff

    idx = np.array(selected, dtype=int)
    return X_train[:, idx], X_test[:, idx], {"selected_features": idx}

In [ ]:
fs_dict = {

    # --------------------------------------------------
    # Baseline
    # --------------------------------------------------
    "none": {
        "function": fs_none,
        "params": {}
    },

    # --------------------------------------------------
    # Filter Methods (fast)
    # --------------------------------------------------
    "mi": {
        "function": fs_mi,
        "params": {"n_features": 128}
    },

    "anova": {
        "function": fs_anova,
        "params": {"n_features": 128}
    },

    "variance": {
        "function": fs_variance,
        "params": {"n_features": 128}
    },

    "pearson": {
        "function": fs_pearson,
        "params": {"n_features": 128}
    },

    "chi2": {
        "function": fs_chi2,
        "params": {"n_features": 128}
    },

    "spearman": {
        "function": fs_spearman,
        "params": {"n_features": 128}
    },

    "entropy": {
        "function": fs_entropy,
        "params": {"n_features": 128, "bins": 10}
    },

    # --------------------------------------------------
    # Wrapper / Embedded (moderate cost)
    # --------------------------------------------------
    "l1": {
        "function": fs_l1,
        "params": {"n_features": 128, "C": 0.1}
    },

    "rf": {
        "function": fs_rf,
        "params": {"n_features": 128, "n_estimators": 200}
    },

    "mrmr": {
        "function": fs_mrmr,
        "params": {"n_features": 128}
    },

    "svmrfe": {
        "function": fs_svmrfe,
        "params": {"n_features": 128}
    },

    "oner": {
        "function": fs_oner,
        "params": {"n_features": 128, "bins": 10}
    },

    # --------------------------------------------------
    # Expensive Methods (be careful)
    # --------------------------------------------------
    "sfs": {
        "function": fs_sfs,
        "params": {"n_features": 32}   # smaller by default (slow)
    },

    "ga": {
        "function": fs_ga,
        "params": {
            "n_features": 64,
            "pop_size": 20,
            "generations": 10,
            "mutation_rate": 0.05
        }
    },

    "cfs": {
        "function": fs_cfs,
        "params": {"k": 64}
    },

    "relief": {
        "function": fs_relief,
        "params": {"n_features": 128}
    }
}

Extraction

In [ ]:
import numpy as np
from sklearn.decomposition import PCA, KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

# For Autoencoder (Keras)
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping


# ==========================================================
# 1) PCA
# ==========================================================

def dr_pca(X_train, y_train, domains_train,
           X_test, domains_test,
           n_components=64):

    reducer = PCA(n_components=n_components)
    Z_train = reducer.fit_transform(X_train)
    Z_test = reducer.transform(X_test)

    return Z_train, Z_test, {"model": reducer}


# ==========================================================
# 2) Kernel PCA (RBF)
# ==========================================================

def dr_kernel_pca(X_train, y_train, domains_train,
                  X_test, domains_test,
                  n_components=64,
                  gamma=None):

    reducer = KernelPCA(
        n_components=n_components,
        kernel="rbf",
        gamma=gamma,
        fit_inverse_transform=False
    )

    Z_train = reducer.fit_transform(X_train)
    Z_test = reducer.transform(X_test)

    return Z_train, Z_test, {"model": reducer}


# ==========================================================
# 3) LDA (Supervised)
# ==========================================================

def dr_lda(X_train, y_train, domains_train,
           X_test, domains_test,
           n_components=1):

    # LDA maximum components = n_classes - 1
    lda = LinearDiscriminantAnalysis(n_components=n_components)

    Z_train = lda.fit_transform(X_train, y_train)
    Z_test = lda.transform(X_test)

    return Z_train, Z_test, {"model": lda}


# ==========================================================
# 4) Autoencoder (Simple Stable Version)
# ==========================================================

def dr_autoencoder(X_train, y_train, domains_train,
                   X_test, domains_test,
                   n_components=64,
                   epochs=50,
                   batch_size=128,
                   random_state=42):

    input_dim = X_train.shape[1]

    # --- Build model ---
    input_layer = layers.Input(shape=(input_dim,))
    encoded = layers.Dense(256, activation="relu")(input_layer)
    encoded = layers.Dense(n_components, activation="linear")(encoded)

    decoded = layers.Dense(256, activation="relu")(encoded)
    decoded = layers.Dense(input_dim, activation="linear")(decoded)

    autoencoder = models.Model(input_layer, decoded)
    encoder = models.Model(input_layer, encoded)

    autoencoder.compile(optimizer="adam", loss="mse")

    # Early stopping for stability
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    # Small validation split (train only)
    autoencoder.fit(
        X_train, X_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[early_stop]
    )

    Z_train = encoder.predict(X_train, verbose=0)
    Z_test = encoder.predict(X_test, verbose=0)

    return Z_train, Z_test, {"model": encoder}

In [ ]:
dr_dict = {

    "pca": {
        "function": dr_pca,
        "params": {"n_components": 64}
    },

    "kernel_pca": {
        "function": dr_kernel_pca,
        "params": {
            "n_components": 64,
            "gamma": None
        }
    },

    "lda": {
        "function": dr_lda,
        "params": {"n_components": 1}
    },

    "autoencoder": {
        "function": dr_autoencoder,
        "params": {
            "n_components": 64,
            "epochs": 50,
            "batch_size": 128
        }
    }
}

Proposed

In [ ]:
# need to create a python lib for this shit

### Models

Classical

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


# ==========================================================
# 1) Logistic Regression
# ==========================================================

def train_logistic_regression(
    X_train, y_train, domains_train=None,
    max_iter=1000,
    C=1.0,
    **kwargs
):

    model = LogisticRegression(
        max_iter=max_iter,
        C=C
    )
    model.fit(X_train, y_train)

    return {
        "model": model,
        "predict_proba": lambda X: model.predict_proba(X)[:, 1],
        "predict": lambda X: model.predict(X)
    }


# ==========================================================
# 2) Random Forest
# ==========================================================

def train_random_forest(
    X_train, y_train, domains_train=None,
    n_estimators=200,
    max_depth=None,
    random_state=42,
    **kwargs
):

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        n_jobs=-1,
        random_state=random_state
    )
    model.fit(X_train, y_train)

    return {
        "model": model,
        "predict_proba": lambda X: model.predict_proba(X)[:, 1],
        "predict": lambda X: model.predict(X)
    }


# ==========================================================
# 3) SVM (RBF)
# ==========================================================

def train_svm(
    X_train, y_train, domains_train=None,
    C=1.0,
    gamma="scale",
    **kwargs
):

    model = SVC(
        kernel="rbf",
        C=C,
        gamma=gamma,
        probability=True
    )
    model.fit(X_train, y_train)

    return {
        "model": model,
        "predict_proba": lambda X: model.predict_proba(X)[:, 1],
        "predict": lambda X: model.predict(X)
    }


# ==========================================================
# 4) Deep Neural Network (ERM)
# ==========================================================

class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)


def train_nn_erm(
    X_train, y_train, domains_train=None,
    hidden_dim=128,
    epochs=50,
    lr=1e-3,
    batch_size=128,
    **kwargs
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

    model = SimpleNN(X_train.shape[1], hidden_dim=hidden_dim).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    def predict_proba(X):
        model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }

In [ ]:
model_dict = {

    "logistic": {
        "function": train_logistic_regression,
        "params": {
            "max_iter": 1000,
            "C": 1.0
        }
    },

    "rf": {
        "function": train_random_forest,
        "params": {
            "n_estimators": 200,
            "max_depth": None
        }
    },

    "svm_rbf": {
        "function": train_svm,
        "params": {
            "C": 1.0,
            "gamma": "scale"
        }
    },

    "nn_erm": {
        "function": train_nn_erm,
        "params": {
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128
        }
    }
}

Regularized

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


# ==========================================================
# L2 Regularized NN
# ==========================================================

def train_nn_l2(
    X_train, y_train, domains_train=None,
    hidden_dim=128,
    epochs=50,
    lr=1e-3,
    batch_size=128,
    weight_decay=1e-4,
    **kwargs
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

    model = SimpleNN(
        input_dim=X_train.shape[1],
        hidden_dim=hidden_dim
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay  # L2 regularization
    )

    criterion = nn.BCEWithLogitsLoss()

    dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    def predict_proba(X):
        model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }
    
    

# ==========================================================
# Dropout Network
# ==========================================================

class DropoutNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout_p=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)


def train_nn_dropout(
    X_train, y_train, domains_train=None,
    hidden_dim=128,
    epochs=50,
    lr=1e-3,
    batch_size=128,
    dropout_p=0.5,
    **kwargs
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

    model = DropoutNN(
        input_dim=X_train.shape[1],
        hidden_dim=hidden_dim,
        dropout_p=dropout_p
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    def predict_proba(X):
        model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }

In [ ]:
model_dict_nn_regularized = {

    "nn_l2": {
        "function": train_nn_l2,
        "params": {
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128,
            "weight_decay": 1e-4
        }
    },

    "nn_dropout": {
        "function": train_nn_dropout,
        "params": {
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128,
            "dropout_p": 0.5
        }
    }
}

Domain aware

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_grl * grad_output, None


class GradientReversal(nn.Module):
    def __init__(self, lambda_grl=1.0):
        super().__init__()
        self.lambda_grl = lambda_grl

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_grl)


class DANN(nn.Module):
    def __init__(self, input_dim, n_domains, hidden_dim=128, lambda_grl=1.0):
        super().__init__()

        # Feature extractor
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # Label predictor
        self.label_classifier = nn.Linear(hidden_dim, 1)

        # Domain classifier
        self.grl = GradientReversal(lambda_grl)
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_domains)
        )

    def forward(self, x):
        features = self.feature_extractor(x)

        label_logits = self.label_classifier(features)

        reversed_features = self.grl(features)
        domain_logits = self.domain_classifier(reversed_features)

        return label_logits, domain_logits
    
    

def train_dann(
    X_train, y_train, domains_train,
    hidden_dim=128,
    epochs=50,
    lr=1e-3,
    batch_size=128,
    lambda_grl=1.0,
    lambda_domain=1.0,
    **kwargs
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Map domains to 0..K-1
    unique_domains = np.unique(domains_train)
    domain_mapping = {d: i for i, d in enumerate(unique_domains)}
    domains_mapped = np.array([domain_mapping[d] for d in domains_train])

    X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
    d_tensor = torch.tensor(domains_mapped, dtype=torch.long).to(device)

    n_domains = len(unique_domains)

    model = DANN(
        input_dim=X_train.shape[1],
        n_domains=n_domains,
        hidden_dim=hidden_dim,
        lambda_grl=lambda_grl
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    label_criterion = nn.BCEWithLogitsLoss()
    domain_criterion = nn.CrossEntropyLoss()

    dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor, d_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()

    for _ in range(epochs):
        for xb, yb, db in loader:
            optimizer.zero_grad()
            label_logits, domain_logits = model(xb)

            label_loss = label_criterion(label_logits, yb)
            domain_loss = domain_criterion(domain_logits, db)

            loss = label_loss + lambda_domain * domain_loss
            loss.backward()
            optimizer.step()

    def predict_proba(X):
        model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits, _ = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }
    
    
def train_vrex(
    X_train, y_train, domains_train,
    hidden_dim=128,
    epochs=50,
    lr=1e-3,
    batch_size=128,
    lambda_vrex=1.0,
    **kwargs
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
    d_tensor = torch.tensor(domains_train, dtype=torch.long).to(device)

    model = SimpleNN(X_train.shape[1], hidden_dim=hidden_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(reduction='none')

    unique_domains = torch.unique(d_tensor)

    model.train()

    for _ in range(epochs):

        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor, d_tensor)
        loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

        for xb, yb, db in loader:

            optimizer.zero_grad()
            logits = model(xb)
            losses = criterion(logits, yb)

            domain_risks = []
            for d in unique_domains:
                mask = (db == d)
                if mask.sum() > 0:
                    domain_risks.append(losses[mask].mean())

            domain_risks = torch.stack(domain_risks)

            mean_risk = domain_risks.mean()
            variance_penalty = domain_risks.var(unbiased=False)

            loss = mean_risk + lambda_vrex * variance_penalty

            loss.backward()
            optimizer.step()

    def predict_proba(X):
        model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }
    
    
from sklearn.svm import SVC
from scipy.stats import mode


class DomainMajoritySVM:
    def __init__(self):
        self.models = []
        self.unique_domains = None

    def fit(self, X_train, y_train, domains_train):
        self.unique_domains = np.unique(domains_train)
        self.models = []

        for d in self.unique_domains:
            mask = domains_train == d
            model_d = SVC(kernel="rbf", probability=True)
            model_d.fit(X_train[mask], y_train[mask])
            self.models.append(model_d)

    def predict(self, X):
        all_preds = np.array([model.predict(X) for model in self.models])
        majority_preds = mode(all_preds, axis=0).mode[0]
        return majority_preds

    def predict_proba(self, X):
        all_probs = np.array([model.predict_proba(X) for model in self.models])
        mean_probs = all_probs.mean(axis=0)
        return mean_probs[:, 1]


def train_domain_majority_svm(
    X_train, y_train, domains_train,
    **kwargs
):

    model = DomainMajoritySVM()
    model.fit(X_train, y_train, domains_train)

    return {
        "model": model,
        "predict_proba": lambda X: model.predict_proba(X),
        "predict": lambda X: model.predict(X)
    }

In [ ]:
model_dict_dg = {

    "dann": {
        "function": train_dann,
        "params": {
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128,
            "lambda_grl": 1.0,
            "lambda_domain": 1.0
        }
    },

    "vrex": {
        "function": train_vrex,
        "params": {
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128,
            "lambda_vrex": 1.0
        }
    },

    "domain_majority_svm": {
        "function": train_domain_majority_svm,
        "params": {}
    }
}

# ADD THE 4 methods (SVM and NN with Vapnik cost)

Vapnik LUSI

In [ ]:
# ============================================================
# Vapnik Exact Predicate Builder (Section 7.1)
# φ(x)=1, φ(x)=x, φ(x)=xx^T
# ============================================================
def build_vapnik_predicates(X):

    n, d = X.shape
    predicates = []

    # 1) phi(x)=1
    predicates.append(np.ones(n))

    # 2) phi(x)=x
    for j in range(d):
        predicates.append(X[:, j])

    # 3) phi(x)=xx^T
    for j in range(d):
        for k in range(j, d):
            predicates.append(X[:, j] * X[:, k])

    Phi = np.column_stack(predicates)

    # Normalize columns for numerical stability
    Phi = Phi / (np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-12)

    return Phi


In [ ]:
# ============================================================
# Predicate-Regularized SVM (Binary, Linear or RBF Kernel)
# If Phi=None or tau=0 → reduces to standard SVM
# ============================================================

import numpy as np
import cvxpy as cp


class PredicateSVM:
    def __init__(self, C=1.0, tau=0.0, kernel="rbf", gamma=1.0):
        """
        C       : SVM regularization parameter
        tau     : predicate strength (tau=0 → standard SVM)
        kernel  : "linear" or "rbf"
        gamma   : RBF parameter
        """
        self.C = C
        self.tau = tau
        self.kernel = kernel
        self.gamma = gamma

        self.alpha = None
        self.b = None
        self.X_train = None
        self.y_train = None

    # --------------------------------------------------------
    # Kernel computation
    # --------------------------------------------------------
    def _compute_kernel(self, X1, X2):
        if self.kernel == "linear":
            return X1 @ X2.T

        elif self.kernel == "rbf":
            X1_sq = np.sum(X1**2, axis=1, keepdims=True)
            X2_sq = np.sum(X2**2, axis=1, keepdims=True)
            sq_dists = X1_sq - 2 * X1 @ X2.T + X2_sq.T
            return np.exp(-self.gamma * sq_dists)

        else:
            raise ValueError("Unsupported kernel.")

    # --------------------------------------------------------
    # Fit
    # --------------------------------------------------------
    def fit(self, X, y, Phi=None):

        self.X_train = X
        self.y_train = y

        n = X.shape[0]

        K = self._compute_kernel(X, X)
        Y = np.outer(y, y)
        Q = Y * K

        # Add predicate term if provided
        if Phi is not None and self.tau > 0:
            P = (Phi @ Phi.T) / Phi.shape[1]
            Q = Q + self.tau * P

        Q = (Q + Q.T) / 2
        Q = cp.psd_wrap(Q)

        alpha = cp.Variable(n)

        objective = cp.Minimize(
            0.5 * cp.quad_form(alpha, Q) - cp.sum(alpha)
        )

        constraints = [
            alpha >= 0,
            alpha <= self.C,
            y @ alpha == 0
        ]

        problem = cp.Problem(objective, constraints)
        problem.solve(solver=cp.OSQP)

        self.alpha = alpha.value

        # Compute bias using support vectors
        support = self.alpha > 1e-5
        K_support = self._compute_kernel(X[support], X)
        self.b = np.mean(
            y[support] - (K_support @ (self.alpha * y))
        )

        return self

    # --------------------------------------------------------
    # Decision function
    # --------------------------------------------------------
    def decision_function(self, X_new):
        K_new = self._compute_kernel(X_new, self.X_train)
        return K_new @ (self.alpha * self.y_train) + self.b

    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------
    def predict(self, X_new):
        return np.sign(self.decision_function(X_new))



def train_predicate_svm(
    X_train, y_train, domains_train=None,
    C=1.0,
    tau=0.0,
    kernel="rbf",
    gamma=1.0,
    use_predicates=True,
    **kwargs
):

    # Convert labels {0,1} → {-1,+1}
    y_svm = 2 * y_train - 1

    # Build predicates if required
    if use_predicates and tau > 0:
        Phi = build_vapnik_predicates(X_train)
    else:
        Phi = None

    model = PredicateSVM(
        C=C,
        tau=tau,
        kernel=kernel,
        gamma=gamma
    )

    model.fit(X_train, y_svm, Phi=Phi)

    def predict_proba(X):
        decision = model.decision_function(X)
        # Convert decision to probability via sigmoid
        probs = 1 / (1 + np.exp(-decision))
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }
    
    
def train_predicate_nn_pipeline(
    X_train, y_train, domains_train=None,
    tau=0.0,
    epochs=200,
    lr=1e-3,
    hidden_dim=32,
    use_predicates=True,
    **kwargs
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(device)

    model = SimpleNN(X_train.shape[1], hidden_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    # Build predicates if required
    if use_predicates and tau > 0:
        Phi = build_vapnik_predicates(X_train)
        Phi_t = torch.tensor(Phi, dtype=torch.float32).to(device)
    else:
        Phi = None

    model.train()

    for _ in range(epochs):

        optimizer.zero_grad()

        logits = model(X_t)
        loss_task = criterion(logits, y_t)

        if Phi is not None and tau > 0:
            preds = torch.sigmoid(logits)

            pred_stat = Phi_t.T @ preds
            true_stat = Phi_t.T @ y_t

            loss_pred = torch.mean((pred_stat - true_stat) ** 2)
            loss = loss_task + tau * loss_pred
        else:
            loss = loss_task

        loss.backward()
        optimizer.step()

    def predict_proba(X):
        model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }

In [ ]:
model_dict_predicate = {

    "predicate_svm_linear": {
        "function": train_predicate_svm,
        "params": {
            "C": 1.0,
            "tau": 0.1,
            "kernel": "linear",
            "gamma": 1.0,
            "use_predicates": True
        }
    },

    "predicate_svm_rbf": {
        "function": train_predicate_svm,
        "params": {
            "C": 1.0,
            "tau": 0.1,
            "kernel": "rbf",
            "gamma": 1.0,
            "use_predicates": True
        }
    },

    "predicate_nn": {
        "function": train_predicate_nn_pipeline,
        "params": {
            "tau": 0.1,
            "epochs": 200,
            "lr": 1e-3,
            "hidden_dim": 32,
            "use_predicates": True
        }
    }
}

No Domain Proposed

In [ ]:
# =========================
# Train SVM Region
# =========================
def train_svm_region(x1, x2):
    if x2.ndim == 1:
        x2 = x2.reshape(-1, 1)
    X = np.hstack([x1, x2])
    svm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
    svm.fit(X)
    return svm


# =========================
# Differentiable SVM Wrapper
# =========================
class SVMPremise(nn.Module):
    def __init__(self, svm_model):
        super().__init__()

        self.support_vectors = torch.tensor(
            svm_model.support_vectors_, dtype=torch.float32
        )
        self.alpha = torch.tensor(
            svm_model.dual_coef_.flatten(), dtype=torch.float32
        )
        self.rho = torch.tensor(
            svm_model.intercept_[0], dtype=torch.float32
        )
        self.gamma = svm_model._gamma

    def rbf_kernel(self, X1, X2):
        sq1 = (X1**2).sum(dim=1, keepdim=True)
        sq2 = (X2**2).sum(dim=1, keepdim=True)
        sqdist = sq1 + sq2.T - 2 * X1 @ X2.T
        return torch.exp(-self.gamma * sqdist)

    def raw_decision(self, x1, x2_pred):
        X = torch.cat([x1, x2_pred], dim=1)
        K = self.rbf_kernel(self.support_vectors, X)
        f = self.alpha @ K + self.rho
        return f

    def forward(self, x1, x2_pred):
        f = self.raw_decision(x1, x2_pred)
        return torch.relu(-f)   # penalty
    
    

def train_nn_svm_premise_global(
    X_train, y_train, domains_train=None,
    mode="combined",  # "plain", "combined", "premise_only"
    lambda_premise=0.1,
    hidden_dim=128,
    epochs=50,
    lr=1e-3,
    batch_size=128,
    **kwargs
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

    dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = SimpleNN(X_train.shape[1], hidden_dim=hidden_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    # -------------------------
    # Single Global SVM
    # -------------------------
    premise = None
    if mode in ["combined", "premise_only"]:
        svm = train_svm_region(X_train, y_train)
        premise = SVMPremise(svm).to(device)

    model.train()

    for _ in range(epochs):
        for xb, yb in loader:

            optimizer.zero_grad()
            logits = model(xb)

            if mode == "plain":
                loss = criterion(logits, yb)

            else:
                premise_loss = premise(xb, logits).mean()

                if mode == "combined":
                    task_loss = criterion(logits, yb)
                    loss = task_loss + lambda_premise * premise_loss
                else:
                    loss = premise_loss

            loss.backward()
            optimizer.step()

    def predict_proba(X):
        model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }

In [ ]:
model_dict_premise_global = {

    "nn_svm_premise_global_combined": {
        "function": train_nn_svm_premise_global,
        "params": {
            "mode": "combined",
            "lambda_premise": 0.1,
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128
        }
    },

    "nn_svm_premise_global_only": {
        "function": train_nn_svm_premise_global,
        "params": {
            "mode": "premise_only",
            "lambda_premise": 0.1,
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128
        }
    }
}

Premise Based

In [ ]:
def train_nn_svm_premise_multidomain(
    X_train, y_train, domains_train,
    mode="combined",  # "plain", "combined", "premise_only"
    lambda_premise=0.1,
    hidden_dim=128,
    epochs=50,
    lr=1e-3,
    batch_size=128,
    **kwargs
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

    dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = SimpleNN(X_train.shape[1], hidden_dim=hidden_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    # -------------------------
    # One SVM per domain
    # -------------------------
    premises = []
    unique_domains = np.unique(domains_train)

    if mode in ["combined", "premise_only"]:
        for d in unique_domains:
            mask = domains_train == d
            svm_d = train_svm_region(X_train[mask], y_train[mask])
            premises.append(SVMPremise(svm_d).to(device))

    n_domains = len(premises)

    model.train()

    for _ in range(epochs):
        for xb, yb in loader:

            optimizer.zero_grad()
            logits = model(xb)

            if mode == "plain":
                loss = criterion(logits, yb)

            else:
                total_premise_loss = 0.0

                for premise in premises:
                    total_premise_loss += premise(xb, logits).mean()

                total_premise_loss /= n_domains

                if mode == "combined":
                    task_loss = criterion(logits, yb)
                    loss = task_loss + lambda_premise * total_premise_loss
                else:
                    loss = total_premise_loss

            loss.backward()
            optimizer.step()

    def predict_proba(X):
        model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return probs

    def predict(X):
        probs = predict_proba(X)
        return (probs >= 0.5).astype(int)

    return {
        "model": model,
        "predict_proba": predict_proba,
        "predict": predict
    }

In [ ]:
model_dict_premise_multidomain = {

    "nn_svm_premise_multidomain_combined": {
        "function": train_nn_svm_premise_multidomain,
        "params": {
            "mode": "combined",
            "lambda_premise": 0.1,
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128
        }
    },

    "nn_svm_premise_multidomain_only": {
        "function": train_nn_svm_premise_multidomain,
        "params": {
            "mode": "premise_only",
            "lambda_premise": 0.1,
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128
        }
    }
}

### Parameters

In [51]:
def select_two_labels(df, label_col="label", labels=(0, 1)):
    """
    Keep only two labels and remap them to 0 and 1.

    labels: tuple/list with exactly two original labels
            First -> mapped to 0
            Second -> mapped to 1
    """

    if len(labels) != 2:
        raise ValueError("Provide exactly two labels.")

    df = df[df[label_col].isin(labels)].copy()

    mapping = {
        labels[0]: 0,
        labels[1]: 1
    }

    label_bin = df[label_col].map(mapping)

    # Find position of label column
    label_pos = df.columns.get_loc(label_col)

    # Insert right after label
    df.insert(label_pos + 1, "label_bin", label_bin)

    return df


In [52]:
df = select_two_labels(df, labels=(0, 1))
df

,subject,session,label,label_bin,time_0_0,time_0_1,time_0_2,time_0_3,time_0_4,time_0_5,...,freq_18_18,freq_18_19,freq_18_20,freq_18_21,freq_19_19,freq_19_20,freq_19_21,freq_20_20,freq_20_21,freq_21_21
2,A01,session1,1,1,3.245072e-11,2.707170e-11,3.160530e-11,3.281442e-11,3.181356e-11,2.884676e-11,...,5.976102e-11,3.050644e-11,6.442204e-12,1.171661e-11,5.464778e-11,1.457287e-11,2.367965e-11,3.652671e-11,5.083133e-11,1.182691e-10
3,A01,session1,0,0,5.061837e-11,4.290988e-11,4.819076e-11,4.964892e-11,4.569411e-11,3.962393e-11,...,8.840444e-11,2.861593e-11,1.735132e-11,2.208399e-11,9.064836e-11,3.220702e-11,4.230613e-11,8.441786e-11,1.219668e-10,2.350288e-10
4,A01,session1,0,0,2.911876e-11,2.147890e-11,2.556846e-11,2.738989e-11,2.745035e-11,2.583514e-11,...,7.598676e-11,3.177676e-11,2.540148e-11,2.644370e-11,7.674789e-11,3.823329e-11,5.740674e-11,7.025289e-11,1.028704e-10,2.139844e-10
5,A01,session1,1,1,4.403216e-11,3.527282e-11,4.161242e-11,4.239562e-11,4.407895e-11,4.003987e-11,...,5.112731e-11,2.979903e-11,1.783687e-12,8.764213e-12,6.918267e-11,2.553461e-11,4.323638e-11,4.968454e-11,6.183407e-11,1.278410e-10
8,A01,session1,1,1,3.807990e-11,3.149787e-11,3.652000e-11,3.858479e-11,3.621218e-11,3.271117e-11,...,7.081198e-11,3.459125e-11,1.170610e-11,1.851671e-11,8.061215e-11,3.431262e-11,4.984833e-11,6.907504e-11,1.026452e-10,2.037375e-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5170,A09,session2,0,0,1.534772e-10,1.136710e-10,1.377245e-10,1.572151e-10,1.586317e-10,1.578940e-10,...,3.439991e-10,2.029151e-10,8.569958e-11,1.450593e-10,3.139110e-10,1.844396e-10,3.298788e-10,3.324392e-10,5.297985e-10,1.040886e-09
5171,A09,session2,0,0,1.120955e-10,8.539854e-11,1.001169e-10,1.131870e-10,1.130774e-10,1.075938e-10,...,5.394434e-10,1.697144e-10,1.761347e-10,3.472588e-10,3.893084e-10,1.933211e-10,3.616828e-10,4.112061e-10,6.932881e-10,1.480112e-09
5172,A09,session2,0,0,9.029920e-11,6.294464e-11,7.717460e-11,8.833137e-11,8.808698e-11,8.097756e-11,...,5.309001e-10,2.546899e-10,1.475707e-10,2.233716e-10,3.326053e-10,9.300899e-11,1.260349e-10,2.242952e-10,3.099867e-10,6.232813e-10
5176,A09,session2,1,1,7.580222e-11,5.965536e-11,7.113222e-11,7.778171e-11,7.332116e-11,6.631644e-11,...,5.237161e-10,2.086888e-10,1.677895e-10,3.444832e-10,3.239231e-10,1.120894e-10,2.429090e-10,3.053000e-10,5.114297e-10,1.050338e-09


### Experiments

Intra Subject

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler


def run_intra_subject_experiment(
    df,
    subject_col="subject",
    session_col="session",
    label_col="label_bin",
    fs_dict=None,
    model_dict=None,
    test_size=0.3,
    random_state=42
):

    results = []

    feature_cols = [
        c for c in df.columns
        if c not in [subject_col, session_col, label_col, "label"]
    ]

    subjects = df[subject_col].unique()

    for subj in subjects:

        df_subj = df[df[subject_col] == subj].copy()

        if df_subj[label_col].nunique() < 2:
            continue

        X = df_subj[feature_cols].values
        y = df_subj[label_col].values

        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=random_state
        )

        for train_idx, test_idx in splitter.split(X, y):

            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            df_train = df_subj.iloc[train_idx]
            df_test  = df_subj.iloc[test_idx]

            domains_train = df_train[session_col].values
            domains_test  = df_test[session_col].values

            if len(np.unique(y_train)) < 2:
                continue

            # ==========================
            # SCALE
            # ==========================
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test  = scaler.transform(X_test)

            for fs_name, fs_cfg in fs_dict.items():

                fs_fn = fs_cfg["function"]
                fs_params = fs_cfg.get("params", {})

                X_train_fs, X_test_fs, _ = fs_fn(
                    X_train, y_train, domains_train,
                    X_test, domains_test,
                    **fs_params
                )

                for model_name, model_cfg in model_dict.items():

                    model_fn = model_cfg["function"]
                    model_params = model_cfg.get("params", {})

                    model = model_fn(
                        X_train_fs, y_train, domains_train,
                        **model_params
                    )

                    y_train_pred = model["predict"](X_train_fs)
                    y_train_prob = model["predict_proba"](X_train_fs)

                    y_test_pred = model["predict"](X_test_fs)
                    y_test_prob = model["predict_proba"](X_test_fs)

                    # Save TRAIN
                    for i in range(len(y_train)):
                        results.append({
                            "subject": subj,
                            "session": df_train.iloc[i][session_col],
                            "split": "train",
                            "fs_method": fs_name,
                            "model": model_name,
                            "y_true": int(y_train[i]),
                            "y_pred": int(y_train_pred[i]),
                            "y_prob": float(y_train_prob[i])
                        })

                    # Save TEST
                    for i in range(len(y_test)):
                        results.append({
                            "subject": subj,
                            "session": df_test.iloc[i][session_col],
                            "split": "test",
                            "fs_method": fs_name,
                            "model": model_name,
                            "y_true": int(y_test[i]),
                            "y_pred": int(y_test_pred[i]),
                            "y_prob": float(y_test_prob[i])
                        })

    return pd.DataFrame(results)


In [61]:
results_df = run_intra_subject_experiment(
    df=df,
    subject_col="subject",
    session_col="session",
    label_col="label_bin",
    fs_dict=fs_dict,
    model_dict=model_dict,
    test_size=0.3,
    random_state=42
)

results_df.head()

,subject,session,split,fs_method,model,y_true,y_pred,y_prob
0,A01,session1,train,none,logistic,0,0,0.071896
1,A01,session2,train,none,logistic,0,0,0.000006
2,A01,session1,train,none,logistic,0,0,0.000103
3,A01,session1,train,none,logistic,1,1,0.998877
4,A01,session2,train,none,logistic,1,1,0.928286


Inter Subject

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def run_inter_subject_experiment(
    df,
    subject_col="subject",
    session_col="session",
    label_col="label_bin",
    fs_dict=None,
    model_dict=None
):
    """
    Inter-subject experiment (LOSO):
    - Leave one subject out for testing
    - Train on remaining subjects
    - Proper scaling (train only)
    - Apply FS + model
    - Save train and test predictions
    """

    results = []

    feature_cols = [
        c for c in df.columns
        if c not in [subject_col, session_col, label_col, "label"]
    ]

    subjects = df[subject_col].unique()

    for test_subject in subjects:

        train_df = df[df[subject_col] != test_subject].copy()
        test_df  = df[df[subject_col] == test_subject].copy()

        ## Skip if test subject has only one class
        #if test_df[label_col].nunique() < 2:
        #    continue

        X_train = train_df[feature_cols].values
        y_train = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        domains_train = train_df[subject_col].values
        domains_test  = test_df[subject_col].values

        ## Safety: skip if train has only one class
        #if len(np.unique(y_train)) < 2:
        #    continue

        # ==========================
        # SCALE (NO LEAKAGE)
        # ==========================
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        for fs_name, fs_cfg in fs_dict.items():

            fs_fn = fs_cfg["function"]
            fs_params = fs_cfg.get("params", {})

            X_train_fs, X_test_fs, _ = fs_fn(
                X_train, y_train, domains_train,
                X_test, domains_test,
                **fs_params
            )

            for model_name, model_cfg in model_dict.items():

                model_fn = model_cfg["function"]
                model_params = model_cfg.get("params", {})

                model = model_fn(
                    X_train_fs, y_train, domains_train,
                    **model_params
                )

                y_train_pred = model["predict"](X_train_fs)
                y_train_prob = model["predict_proba"](X_train_fs)

                y_test_pred = model["predict"](X_test_fs)
                y_test_prob = model["predict_proba"](X_test_fs)

                # Save TRAIN
                for i in range(len(y_train)):
                    results.append({
                        "subject": train_df.iloc[i][subject_col],
                        "session": train_df.iloc[i][session_col],
                        "split": "train",
                        "fs_method": fs_name,
                        "model": model_name,
                        "y_true": int(y_train[i]),
                        "y_pred": int(y_train_pred[i]),
                        "y_prob": float(y_train_prob[i]),
                        "test_subject": test_subject
                    })

                # Save TEST
                for i in range(len(y_test)):
                    results.append({
                        "subject": test_subject,
                        "session": test_df.iloc[i][session_col],
                        "split": "test",
                        "fs_method": fs_name,
                        "model": model_name,
                        "y_true": int(y_test[i]),
                        "y_pred": int(y_test_pred[i]),
                        "y_prob": float(y_test_prob[i]),
                        "test_subject": test_subject
                    })

    return pd.DataFrame(results)


In [63]:
results_inter_subject = run_inter_subject_experiment(
    df=df,
    subject_col="subject",
    session_col="session",
    label_col="label_bin",
    fs_dict=fs_dict,
    model_dict=model_dict
)

results_inter_subject.head()

,subject,session,split,fs_method,model,y_true,y_pred,y_prob,test_subject
0,A02,session1,train,none,logistic,0,0,0.414773,A01
1,A02,session1,train,none,logistic,1,0,0.428407,A01
2,A02,session1,train,none,logistic,1,0,0.397324,A01
3,A02,session1,train,none,logistic,0,0,0.428962,A01
4,A02,session1,train,none,logistic,1,1,0.529556,A01


In [67]:
from sklearn.metrics import accuracy_score
import pandas as pd

def compute_simple_accuracy(results_df):
    
    rows = []
    
    for (fs, model), group in results_df.groupby(["fs_method", "model"]):
        
        train_df = group[group["split"] == "train"]
        test_df  = group[group["split"] == "test"]
        
        train_acc = accuracy_score(train_df["y_true"], train_df["y_pred"])
        test_acc  = accuracy_score(test_df["y_true"], test_df["y_pred"])
        
        rows.append({
            "fs_method": fs,
            "model": model,
            "train_acc": train_acc,
            "test_acc": test_acc
        })
    
    return pd.DataFrame(rows).sort_values("test_acc", ascending=False)

accuracy_table = compute_simple_accuracy(results_inter_subject)
accuracy_table


,fs_method,model,train_acc,test_acc
0,none,logistic,0.722078,0.627701
1,none,mlp,0.676939,0.623071
2,top_var_20,logistic,0.645255,0.604938
3,top_var_20,mlp,0.595197,0.563657
